In [7]:
import mne
import numpy as np
import pandas as pd
import os
from tabulate import tabulate # Librería para organizar tablas
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:


# --- 1. Definición de la función RMS solicitada ---
def calcular_rms_personalizado(data_epocas):
    # RMS por época y canal sobre el eje del tiempo (axis=2)
    rms_por_epoca = np.sqrt(np.mean(data_epocas**2, axis=2))
    # Promedio de RMS a través de todas las épocas (axis=0)
    rms_promedio_sujeto = np.mean(rms_por_epoca, axis=0)
    return rms_promedio_sujeto

# --- 2. Configuración de datos ---
sujetos = [f'S0{i:02d}' for i in range(14, 24)] 
runs = ['R04', 'R08', 'R12']
path_carpeta = 'sujetos/' 

rms_final_izq = []
rms_final_der = []

# --- 3. Bucle de extracción de datos ---
print("Iniciando procesamiento de sujetos...")

for s in sujetos:
    epocas_izq_sujeto = []
    epocas_der_sujeto = []
    
    for r in runs:
        archivo = os.path.join(path_carpeta, f"{s}{r}.edf")
        
        # Cargar señal
        raw = mne.io.read_raw_edf(archivo, preload=True, verbose=False)
        raw.rename_channels(lambda x: x.strip('.'))
        
        # Filtro de banda (8-30 Hz)
        raw.filter(8., 30., fir_design='firwin', verbose=False)
        
        # Extraer eventos
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        
        # Segmentación
        epochs = mne.Epochs(raw, events, event_id={'T1': event_id['T1'], 'T2': event_id['T2']}, 
                            tmin=0.5, tmax=3.5, baseline=None, preload=True, verbose=False)
        
        epocas_izq_sujeto.append(epochs['T1'].get_data(copy=True))
        epocas_der_sujeto.append(epochs['T2'].get_data(copy=True))

    # Concatenar épocas y calcular RMS promedio por sujeto
    data_total_izq = np.concatenate(epocas_izq_sujeto, axis=0)
    data_total_der = np.concatenate(epocas_der_sujeto, axis=0)
    
    rms_final_izq.append(calcular_rms_personalizado(data_total_izq))
    rms_final_der.append(calcular_rms_personalizado(data_total_der))
    
    print(f"-> Sujeto {s} completado.")

# --- Punto 2: Construcción y Visualización de DataFrames ---
nombres_canales = raw.ch_names

# DataFrame Mano Izquierda
df_izquierdo = pd.DataFrame(rms_final_izq, columns=nombres_canales, index=sujetos)

# DataFrame Mano Derecha
df_derecho = pd.DataFrame(rms_final_der, columns=nombres_canales, index=sujetos)

# Visualización organizada con Tabulate
# Nota: Mostramos solo los primeros 10 canales para que la tabla no se rompa por el ancho
print("\n" + "="*30)
print(" GRUPO MANO IZQUIERDA (T1) ")
print("="*30)
print(tabulate(df_izquierdo.iloc[:, :10], headers='keys', tablefmt='fancy_grid', numalign="center"))

print("\n" + "="*30)
print(" GRUPO MANO DERECHA (T2) ")
print("="*30)
print(tabulate(df_derecho.iloc[:, :10], headers='keys', tablefmt='fancy_grid', numalign="center"))

Iniciando procesamiento de sujetos...
-> Sujeto S014 completado.
-> Sujeto S015 completado.
-> Sujeto S016 completado.
-> Sujeto S017 completado.
-> Sujeto S018 completado.
-> Sujeto S019 completado.
-> Sujeto S020 completado.
-> Sujeto S021 completado.
-> Sujeto S022 completado.
-> Sujeto S023 completado.

 GRUPO MANO IZQUIERDA (T1) 
╒══════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╕
│      │     Fc5     │     Fc3     │     Fc1     │     Fcz     │     Fc2     │     Fc4     │     Fc6     │     C5      │     C3      │     C1      │
╞══════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ S014 │ 1.80832e-05 │ 1.76988e-05 │ 1.78402e-05 │ 1.77809e-05 │ 1.77373e-05 │ 1.6069e-05  │ 1.40361e-05 │ 1.8889e-05  │ 1.73758e-05 │ 1.79005e-05 │
├──────┼─────────────┼─────────────┼─────────────┼─────────────┼───

In [12]:


# --- 1. FUNCIÓN DE EVALUACIÓN ESTADÍSTICA ---
def realizar_analisis_estadistico(data_izq, data_der):
    """
    Evalúa Normalidad y Homocedasticidad para decidir entre 
    T-Student o Mann-Whitney.
    """
    # Prueba de Normalidad (Shapiro-Wilk)
    # p > 0.05 significa que los datos son NORMALES
    _, p_shapiro_izq = stats.shapiro(data_izq)
    _, p_shapiro_der = stats.shapiro(data_der)
    
    # Prueba de Homocedasticidad (Levene)
    # p > 0.05 significa que las varianzas son IGUALES
    _, p_levene = stats.levene(data_izq, data_der)
    
    # Verificación de supuestos
    es_normal = (p_shapiro_izq > 0.05) and (p_shapiro_der > 0.05)
    es_homocedastico = (p_levene > 0.05)
    cumple_supuestos = es_normal and es_homocedastico
    
    # Selección de la prueba de hipótesis
    if cumple_supuestos:
        # Se cumplen ambos: T-Student para muestras independientes
        metodo = 'T-Student (Paramétrica)'
        _, p_valor = stats.ttest_ind(data_izq, data_der)
    else:
        # No se cumple alguno: U de Mann-Whitney (No Paramétrica)
        metodo = 'Mann-Whitney (No Paramétrica)'
        _, p_valor = stats.mannwhitneyu(data_izq, data_der)
        
    return {
        'p_sh_izq': p_shapiro_izq,
        'p_sh_der': p_shapiro_der,
        'p_levene': p_levene,
        'cumple': 'SÍ' if cumple_supuestos else 'NO',
        'metodo': metodo,
        'p_final': p_valor
    }

# --- 2. EJECUCIÓN DEL ANÁLISIS POR CANAL ---
resultados_detallados = []

# Limpiamos la lista antes de empezar para evitar duplicados
for canal in nombres_canales:
    d_izq = df_izquierdo[canal].values
    d_der = df_derecho[canal].values
    
    # Llamamos a nuestra función
    res = realizar_analisis_estadistico(d_izq, d_der)
    
    resultados_detallados.append({
        'Canal': canal,
        'Norm. Izq (p)': round(res['p_sh_izq'], 4),
        'Norm. Der (p)': round(res['p_sh_der'], 4),
        'Homoced. (p)': round(res['p_levene'], 4),
        '¿Cumple Supuestos?': res['cumple'],
        'Prueba Usada': res['metodo'],
        'P-Valor Final': round(res['p_final'], 6),
        'Significativo': 'SÍ' if res['p_final'] < 0.05 else 'NO'
    })

# --- 3. CREACIÓN DEL REPORTE Y VISUALIZACIÓN ---
df_reporte_estat = pd.DataFrame(resultados_detallados)

print("\n" + "="*95)
print(" REPORTE POBLACIONAL: VALIDACIÓN DE SUPUESTOS Y PRUEBAS DE HIPÓTESIS ")
print("="*95)
# Mostramos una muestra de los datos (ajustar .head() si quieres ver más)
print(tabulate(df_reporte_estat.head(15), headers='keys', tablefmt='fancy_grid', showindex=False))

# Canales significativos
canales_clave = df_reporte_estat[df_reporte_estat['Significativo'] == 'SÍ'].sort_values(by='P-Valor Final')

if not canales_clave.empty:
    print("\n" + "*"*50)
    print(" CANALES CON DIFERENCIA SIGNIFICATIVA IDENTIFICADOS ")
    print("*"*50)
    print(tabulate(canales_clave, headers='keys', tablefmt='fancy_grid', showindex=False))
    
    # Generar Boxplot del canal con menor p-valor
    mejor_canal = canales_clave.iloc[0]['Canal']
    
    plot_data = pd.DataFrame({
        'RMS': np.concatenate([df_izquierdo[mejor_canal], df_derecho[mejor_canal]]),
        'Tarea': ['Mano Izquierda (T1)']*10 + ['Mano Derecha (T2)']*10
    })
    
    plt.figure(figsize=(10, 7))
    sns.boxplot(x='Tarea', y='RMS', data=plot_data, palette='Set2', width=0.4)
    sns.swarmplot(x='Tarea', y='RMS', data=plot_data, color=".2", size=9)
    
    p_rep = canales_clave.iloc[0]['P-Valor Final']
    plt.title(f'Distribución de RMS en Canal {mejor_canal}\n(p-valor = {p_rep:.6f})', fontsize=14)
    plt.ylabel('Potencia RMS Promedio', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.show()
else:
    print("\n[!] No se encontraron canales con diferencia estadística significativa.")


 REPORTE POBLACIONAL: VALIDACIÓN DE SUPUESTOS Y PRUEBAS DE HIPÓTESIS 
╒═════════╤═════════════════╤═════════════════╤════════════════╤══════════════════════╤═════════════════════════╤═════════════════╤═════════════════╕
│ Canal   │   Norm. Izq (p) │   Norm. Der (p) │   Homoced. (p) │ ¿Cumple Supuestos?   │ Prueba Usada            │   P-Valor Final │ Significativo   │
╞═════════╪═════════════════╪═════════════════╪════════════════╪══════════════════════╪═════════════════════════╪═════════════════╪═════════════════╡
│ Fc5     │          0.1766 │          0.2806 │         0.9541 │ SÍ                   │ T-Student (Paramétrica) │        0.985152 │ NO              │
├─────────┼─────────────────┼─────────────────┼────────────────┼──────────────────────┼─────────────────────────┼─────────────────┼─────────────────┤
│ Fc3     │          0.8747 │          0.7454 │         0.9075 │ SÍ                   │ T-Student (Paramétrica) │        0.981147 │ NO              │
├─────────┼─────────────────┼

In [16]:
import mne
import os
import numpy as np
import pandas as pd
from scipy import stats
from tabulate import tabulate

# --- 1. Configuración de Rutas Relativas ---
path_carpeta = 'sujetos2' 
sujeto = "sub-014" 
runs = ["4", "8", "12"]

epocas_izq_sujeto = []
epocas_der_sujeto = []

print(f"--- Procesando Dataset OpenNeuro ds004362 | Sujeto: {sujeto} ---")

for r in runs:
    nombre_archivo = f"{sujeto}_task-motion_run-{r}_eeg.set"
    ruta_completa = os.path.join(path_carpeta, nombre_archivo)
    
    if not os.path.exists(ruta_completa):
        print(f"❌ Archivo no encontrado: {nombre_archivo}")
        continue
        
    # Cargar y pre-procesar
    raw = mne.io.read_raw_eeglab(ruta_completa, preload=True, verbose=False)
    raw.filter(8., 30., fir_design='firwin', verbose=False)
    
    # Extraer eventos
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    
    # --- MAPEADOR DE EVENTOS SEGURO ---
    # En OpenNeuro/EEGLAB, T1 puede aparecer como 'T1' o como '2'
    # T2 puede aparecer como 'T2' o como '3' (según la codificación interna)
    mapping = {}
    for clave in ['T1', '1', '2']: # Posibles nombres para mano izquierda
        if clave in event_id:
            mapping['T1'] = event_id[clave]
            break
            
    for clave in ['T2', '2', '3']: # Posibles nombres para mano derecha
        if clave in event_id:
            mapping['T2'] = event_id[clave]
            break

    if 'T1' not in mapping or 'T2' not in mapping:
        print(f"⚠️ No se hallaron marcadores en {nombre_archivo}. Detectados: {event_id}")
        continue

    # Segmentación (Épocas de 3 segundos)
    epochs = mne.Epochs(raw, events, event_id=mapping, 
                        tmin=0.5, tmax=3.5, baseline=None, preload=True, verbose=False)
    
    epocas_izq_sujeto.append(epochs['T1'].get_data(copy=True))
    epocas_der_sujeto.append(epochs['T2'].get_data(copy=True))

# --- 2. Análisis Estadístico de Comparación ---
if epocas_izq_sujeto:
    # Unión de épocas (Análisis Intra-sujeto)
    data_izq = np.concatenate(epocas_izq_sujeto, axis=0)
    data_der = np.concatenate(epocas_der_sujeto, axis=0)

    # Cálculo de RMS por época: (épocas, canales)
    rms_izq = np.sqrt(np.mean(data_izq**2, axis=2))
    rms_der = np.sqrt(np.mean(data_der**2, axis=2))

    nombres_canales = raw.ch_names
    reporte = []

    for i, canal in enumerate(nombres_canales):
        v_izq, v_der = rms_izq[:, i], rms_der[:, i]
        
        # Validación de Supuestos
        p_norm = min(stats.shapiro(v_izq)[1], stats.shapiro(v_der)[1])
        p_homoced = stats.levene(v_izq, v_der)[1]
        
        cumple = p_norm > 0.05 and p_homoced > 0.05
        
        # Selección de Prueba
        if cumple:
            metodo = 'T-Student'
            p_val = stats.ttest_ind(v_izq, v_der)[1]
        else:
            metodo = 'Mann-Whitney'
            p_val = stats.mannwhitneyu(v_izq, v_der)[1]
            
        reporte.append([canal, round(p_norm, 4), round(p_homoced, 4), 
                        'SÍ' if cumple else 'NO', metodo, round(p_val, 6), 
                        'SÍ' if p_val < 0.05 else 'NO'])

    # --- 3. Resultados Finales ---
    headers = ["Canal", "Norm (p)", "Homoced (p)", "¿Supuestos?", "Prueba", "P-Valor", "Sig."]
    print("\n" + "✅" + "="*95)
    print(f" REPORTE ESTADÍSTICO FINAL | SUJETO 14 | DATASET: ds004362 ")
    print("="*95)
    print(tabulate(reporte[:20], headers=headers, tablefmt='fancy_grid'))
else:
    print("❌ Fallo en la extracción de datos. Revisa las etiquetas de eventos.")

--- Procesando Dataset OpenNeuro ds004362 | Sujeto: sub-014 ---
⚠️ No se hallaron marcadores en sub-014_task-motion_run-4_eeg.set. Detectados: {np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
⚠️ No se hallaron marcadores en sub-014_task-motion_run-8_eeg.set. Detectados: {np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
⚠️ No se hallaron marcadores en sub-014_task-motion_run-12_eeg.set. Detectados: {np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
❌ Fallo en la extracción de datos. Revisa las etiquetas de eventos.
